In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
models ={
    'Image Backbone': ['../../models/BTSv2_PSonly/stoic-sweep-6'],
    'Oracle-2 Lite': ['../../models/BTSv2-lite/gentle-sweep-1', 
                      '../../models/BTSv2-lite/comfy-smoke-128',
                      '../../models/BTSv2-lite/deft-monkey-129',
                      '../../models/BTSv2-lite/flowing-feather-130',
                      '../../models/BTSv2-lite/stilted-dawn-131'],
    'Oracle-2': ['../../models/BTSv2/peachy-sweep-4',
                 '../../models/BTSv2/rare-night-119', 
                 '../../models/BTSv2/winter-meadow-120', 
                 '../../models/BTSv2/rural-tree-121',
                 '../../models/BTSv2/resilient-feather-125'],
    'Oracle-2 Omni': ['../../models/BTSv2-pro/woven-sweep-9',
                      '../../models/BTSv2-pro/smooth-haze-141',
                      '../../models/BTSv2-pro/splendid-sun-142',
                      '../../models/BTSv2-pro/sweet-glade-143',
                      '../../models/BTSv2-pro/dazzling-thunder-144'],
}

colors = {
    'Oracle-2 Lite': '#994882',
    'Oracle-2': '#008080',
    'Oracle-2 Omni': '#FF6645',
    'Image Backbone': "#000000",
}

markers = {
    'Oracle-2 Lite': 's',
    'Oracle-2': 'D',
    'Oracle-2 Omni': 'o',
    'Image Backbone': '',
}

linestyles = {
    'Oracle-2 Lite': 'solid',
    'Oracle-2': 'solid',
    'Oracle-2 Omni': 'solid',
    'Image Backbone': 'dotted',
}

In [3]:
days = 2**np.arange(0, 11)

In [4]:
def get_df(model_path, depth, day):


    if model_path in models['Image Backbone']:
        path = f"{model_path}/reports/depth{depth}/report_trigger+1.csv"
    else:
        path = f"{model_path}/reports/depth{depth}/report_trigger+{day}.csv"
    df = pd.read_csv(path)
    return df
        

In [5]:
def get_tables_for_all_models(model_choice, depth, day):

    tables = []
    cols = None
    names = None

    for model_path in models[model_choice]:
        df = get_df(model_path, depth, day)
        tables.append(df.apply(pd.to_numeric, errors='coerce').to_numpy()[:, 1:])
        cols = df.columns[1:]
        names = df['Class'].to_numpy()

    # find the mean and std for each metric across the 5 runs
    mean_table = np.mean(tables, axis=0)
    std_table = np.std(tables, axis=0)

    # add the columns and rows back from the first table
    new_df = pd.DataFrame()

    new_df['Class'] = names
    for i, col in enumerate(cols):

        # add the mean +/- std as a sring
        new_df[col] = [f"{mean_table[j, i]:.2f}±{std_table[j, i]:.2f}" for j in range(mean_table.shape[0])]

    return new_df
        
    

In [6]:
def get_f1_scores(model_choice, depth, days=[1,8,128]):

    tables = []

    for d in days:
        df = get_tables_for_all_models(model_choice, depth, d)
        
        # rename column to include the days like f1_d
        df.rename(columns={'f1-score': f'{model_choice} F1_{{{d}}}'}, inplace=True)
        df.drop(columns=['precision', 'recall', 'support'], inplace=True)

        tables.append(df)

    tables = [df.set_index('Class') for df in tables]

    # join the tables on the class
    return pd.concat(tables, axis=1)


In [ ]:
def get_model_comparison_f1(depth, days=[1,8, 128]):

    tables = []
    for model in models:

        if model=='Image Backbone':
            continue
    
        df = get_f1_scores(model, depth, days)
        tables.append(df)

    # join the tables on the class
    return pd.concat(tables, axis=1)

In [8]:
get_model_comparison_f1(2)

,Oracle-2 Lite F1_{1},Oracle-2 Lite F1_{8},Oracle-2 Lite F1_{128},Oracle-2 Lite F1_{1024},Oracle-2 F1_{1},Oracle-2 F1_{8},Oracle-2 F1_{128},Oracle-2 F1_{1024},Oracle-2 Omni F1_{1},Oracle-2 Omni F1_{8},Oracle-2 Omni F1_{128},Oracle-2 Omni F1_{1024}
Class,,,,,,,,,,,,
AGN,0.56±0.01,0.71±0.02,0.74±0.01,0.79±0.02,0.91±0.01,0.94±0.01,0.95±0.01,0.95±0.01,0.93±0.00,0.95±0.00,0.96±0.00,0.96±0.00
CV,0.31±0.01,0.52±0.02,0.73±0.02,0.78±0.02,0.74±0.01,0.81±0.03,0.91±0.01,0.93±0.01,0.87±0.01,0.89±0.01,0.94±0.01,0.95±0.01
SLSN,0.01±0.01,0.05±0.02,0.17±0.05,0.21±0.04,0.09±0.01,0.11±0.02,0.28±0.06,0.32±0.06,0.22±0.04,0.28±0.05,0.42±0.02,0.43±0.02
SN-II,0.10±0.04,0.29±0.03,0.65±0.01,0.69±0.01,0.28±0.03,0.34±0.04,0.68±0.01,0.70±0.01,0.42±0.01,0.48±0.01,0.72±0.01,0.72±0.01
SN-Ia,0.13±0.03,0.67±0.02,0.87±0.01,0.87±0.01,0.45±0.04,0.66±0.01,0.86±0.02,0.86±0.02,0.68±0.02,0.75±0.01,0.89±0.01,0.89±0.00
SN-Ib/c,0.03±0.01,0.12±0.02,0.29±0.03,0.30±0.03,0.11±0.01,0.17±0.01,0.27±0.04,0.25±0.06,0.10±0.02,0.13±0.03,0.21±0.04,0.22±0.03
Varstar,0.16±0.01,0.26±0.01,0.40±0.01,0.44±0.02,0.83±0.01,0.86±0.03,0.89±0.02,0.88±0.02,0.93±0.01,0.94±0.00,0.94±0.01,0.95±0.01
accuracy,0.26±0.01,0.53±0.01,0.72±0.01,0.74±0.01,0.54±0.02,0.66±0.01,0.82±0.01,0.83±0.01,0.69±0.01,0.74±0.00,0.86±0.00,0.86±0.00
macro avg,0.19±0.01,0.38±0.01,0.55±0.01,0.58±0.01,0.49±0.01,0.56±0.01,0.69±0.01,0.70±0.01,0.59±0.01,0.63±0.01,0.73±0.01,0.73±0.01


In [9]:
print(get_model_comparison_f1(2).to_latex())

\begin{tabular}{lllllllllllll}
\toprule
 & Oracle-2 Lite F1_{1} & Oracle-2 Lite F1_{8} & Oracle-2 Lite F1_{128} & Oracle-2 Lite F1_{1024} & Oracle-2 F1_{1} & Oracle-2 F1_{8} & Oracle-2 F1_{128} & Oracle-2 F1_{1024} & Oracle-2 Omni F1_{1} & Oracle-2 Omni F1_{8} & Oracle-2 Omni F1_{128} & Oracle-2 Omni F1_{1024} \\
Class &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
AGN & 0.56±0.01 & 0.71±0.02 & 0.74±0.01 & 0.79±0.02 & 0.91±0.01 & 0.94±0.01 & 0.95±0.01 & 0.95±0.01 & 0.93±0.00 & 0.95±0.00 & 0.96±0.00 & 0.96±0.00 \\
CV & 0.31±0.01 & 0.52±0.02 & 0.73±0.02 & 0.78±0.02 & 0.74±0.01 & 0.81±0.03 & 0.91±0.01 & 0.93±0.01 & 0.87±0.01 & 0.89±0.01 & 0.94±0.01 & 0.95±0.01 \\
SLSN & 0.01±0.01 & 0.05±0.02 & 0.17±0.05 & 0.21±0.04 & 0.09±0.01 & 0.11±0.02 & 0.28±0.06 & 0.32±0.06 & 0.22±0.04 & 0.28±0.05 & 0.42±0.02 & 0.43±0.02 \\
SN-II & 0.10±0.04 & 0.29±0.03 & 0.65±0.01 & 0.69±0.01 & 0.28±0.03 & 0.34±0.04 & 0.68±0.01 & 0.70±0.01 & 0.42±0.01 & 0.48±0.01 & 0.72±0.01 & 0.72±0.01 \\
SN-Ia & 0.13±0.03 & 0.67±

In [10]:
print(get_model_comparison_f1(1).to_latex())

\begin{tabular}{lllllllllllll}
\toprule
 & Oracle-2 Lite F1_{1} & Oracle-2 Lite F1_{8} & Oracle-2 Lite F1_{128} & Oracle-2 Lite F1_{1024} & Oracle-2 F1_{1} & Oracle-2 F1_{8} & Oracle-2 F1_{128} & Oracle-2 F1_{1024} & Oracle-2 Omni F1_{1} & Oracle-2 Omni F1_{8} & Oracle-2 Omni F1_{128} & Oracle-2 Omni F1_{1024} \\
Class &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
Persistent & 0.60±0.01 & 0.81±0.01 & 0.89±0.00 & 0.93±0.00 & 0.90±0.00 & 0.93±0.00 & 0.96±0.00 & 0.97±0.00 & 0.94±0.00 & 0.96±0.00 & 0.97±0.00 & 0.98±0.00 \\
Transient & 0.57±0.04 & 0.88±0.00 & 0.94±0.00 & 0.96±0.00 & 0.94±0.00 & 0.96±0.00 & 0.98±0.00 & 0.98±0.00 & 0.97±0.00 & 0.98±0.00 & 0.98±0.00 & 0.99±0.00 \\
accuracy & 0.58±0.02 & 0.85±0.01 & 0.92±0.00 & 0.95±0.00 & 0.93±0.00 & 0.95±0.00 & 0.97±0.00 & 0.98±0.00 & 0.96±0.00 & 0.97±0.00 & 0.98±0.00 & 0.98±0.00 \\
macro avg & 0.58±0.02 & 0.84±0.01 & 0.91±0.00 & 0.95±0.00 & 0.92±0.00 & 0.95±0.00 & 0.97±0.00 & 0.98±0.00 & 0.96±0.00 & 0.97±0.00 & 0.98±0.00 & 0.98±0.00 \\
wei